# Data loading, featurization & round-trip

Loads QM9 and ZINC-250k from **`torch_molecule`** (HuggingFace), featurizes to dense
`(X, E)` tensors via the `dataset` package, and reports how much each drops.

- **QM9** — neutral SMILES + DFT targets (`mu, alpha, homo, lumo, gap, cv`); round-trip *report-only*.
- **ZINC** — keeps `N+`/`O-`; `charge_aware=True`; round-trip filter *applied*; targets (`logP, qed, SAS`) recomputed via RDKit.

> Old GDB-9 SDF / chemical_vae loaders live in git history at `cb2fd22` — see `DATA_PROVENANCE.md` to revert.

## Setup

In [1]:
import os

REPO = "flow-matching-molecules"
if not os.path.isdir(REPO):
    !git clone https://github.com/Nico-Conti/flow-matching-molecules.git
os.chdir(REPO if os.path.basename(os.getcwd()) != REPO else ".")

!pip install -q uv
!uv pip install --system -q -e .
print("cwd:", os.getcwd())

cwd: /content/flow-matching-molecules


## 1 · Load QM9

In [2]:
import numpy as np
from dataset import load_qm9, load_zinc, MoleculeDataset, collate_dense

LIMIT = None   # set None for the full datasets

qm9 = load_qm9(local_dir="data", limit=LIMIT)   # default targets = EDM six
print("QM9 :", len(qm9["X"]), "molecules | y", qm9["y"].shape, "| targets", qm9["targets"])
print("  stats:", qm9["stats"])

Found existing file at data/qm9.csv
QM9 : 132040 molecules | y (132040, 6) | targets ('mu', 'alpha', 'homo', 'lumo', 'gap', 'cv')
  stats: {'kept': 132040, 'drop_vocab': 1845}


## 2 · Load ZINC-250k

In [3]:
zinc = load_zinc(local_dir="data", limit=LIMIT)   # default targets = logP, qed, SAS
print("ZINC:", len(zinc["X"]), "molecules | y", zinc["y"].shape, "| targets", zinc["targets"])
print("  stats:", zinc["stats"])

Found existing file at data/zinc250k.csv.gz
ZINC: 247449 molecules | y (247449, 3) | targets ('logP', 'qed', 'SAS')
  stats: {'kept': 247449, 'drop_vocab': 2005, 'drop_roundtrip': 1}


## 3 · Featurizer round-trip checks
`smiles -> (X, E) -> smiles` should reproduce the canonical molecule.

In [4]:
from rdkit import Chem
from dataset import smiles_to_tensor, tensor_to_mol, QM9_ATOMS, ZINC_ATOMS

def canon(s):
    return Chem.MolToSmiles(Chem.MolFromSmiles(s), isomericSmiles=False)

def check(s, vocab, charge_aware):
    X, E = smiles_to_tensor(s, atom_vocab=vocab, charge_aware=charge_aware)
    m = tensor_to_mol(X, E, atom_vocab=vocab)
    got = Chem.MolToSmiles(m, isomericSmiles=False) if m is not None else None
    print(f"  {s:22s} -> {str(got):22s} match={m is not None and got == canon(s)}")

print("QM9 (charge_aware, neutral):")
for s in ["CCO", "CC(=O)O", "c1ccncc1", "N#Cc1ccccc1"]:
    check(s, QM9_ATOMS, charge_aware=True)
print("ZINC (charge-aware):")
for s in ["CC[N+](C)(C)C", "CC(=O)[O-]", "Clc1ccccc1", "c1ccc2ccccc2c1"]:
    check(s, ZINC_ATOMS, charge_aware=True)

QM9 (charge_aware, neutral):
  CCO                    -> CCO                    match=True
  CC(=O)O                -> CC(=O)O                match=True
  c1ccncc1               -> c1ccncc1               match=True
  N#Cc1ccccc1            -> N#Cc1ccccc1            match=True
ZINC (charge-aware):
  CC[N+](C)(C)C          -> CC[N+](C)(C)C          match=True
  CC(=O)[O-]             -> CC(=O)[O-]             match=True
  Clc1ccccc1             -> Clc1ccccc1             match=True
  c1ccc2ccccc2c1         -> c1ccc2ccccc2c1         match=True


## 4 · Round-trip drop summary
`kept` = round-trips exactly; `kept_no_roundtrip` = featurized but not gated (QM9);
`drop_*` = excluded (vocab / round-trip / parse).

In [5]:
def show(name, stats):
    n = sum(stats.values())
    kept = stats.get("kept", 0) + stats.get("kept_no_roundtrip", 0)
    print(f"{name}: {kept:,}/{n:,} usable ({kept/n:.2%})")
    for k, v in sorted(stats.items()):
        print(f"    {k:20s} {v:>6,} ({v/n:.2%})")

show("QM9 ", qm9["stats"])
print()
show("ZINC", zinc["stats"])

QM9 : 132,040/133,885 usable (98.62%)
    drop_vocab            1,845 (1.38%)
    kept                 132,040 (98.62%)

ZINC: 247,449/249,455 usable (99.20%)
    drop_roundtrip            1 (0.00%)
    drop_vocab            2,005 (0.80%)
    kept                 247,449 (99.20%)


## 6 · Targets present & per-property normalization stats
Confirms `y` is aligned (one row per usable molecule) and has no missing values,
then reports mean/std per property — the conditioning-normalization stats for training.

In [6]:
def target_report(name, d):
    y = np.asarray(d["y"])
    assert y.shape[0] == len(d["X"]), f"{name}: y rows {y.shape[0]} != #mols {len(d['X'])}"
    n_nan = int(np.isnan(y).sum())
    print(f"{name}: y {y.shape} | targets {d['targets']} | NaNs {n_nan}")
    for j, t in enumerate(d["targets"]):
        c = y[:, j]
        print(f"    {t:6s} mean={c.mean():+.4f}  std={c.std():.4f}  min={c.min():+.4f}  max={c.max():+.4f}")
    return {t: (float(y[:, j].mean()), float(y[:, j].std())) for j, t in enumerate(d["targets"])}

norm_stats = {"qm9": target_report("QM9 ", qm9), "zinc": target_report("ZINC", zinc)}
norm_stats

QM9 : y (132040, 6) | targets ('mu', 'alpha', 'homo', 'lumo', 'gap', 'cv') | NaNs 0
    mu     mean=+2.6497  std=1.4013  min=+0.0000  max=+9.4219
    alpha  mean=+75.2282  std=8.1766  min=+6.3100  max=+143.5300
    homo   mean=-0.2402  std=0.0218  min=-0.4286  max=-0.1325
    lumo   mean=+0.0114  std=0.0469  min=-0.1750  max=+0.1935
    gap    mean=+0.2516  std=0.0472  min=+0.0376  max=+0.6221
    cv     mean=+31.5946  std=4.0670  min=+6.0020  max=+46.9690
ZINC: y (247449, 3) | targets ('logP', 'qed', 'SAS') | NaNs 0
    logP   mean=+2.4554  std=1.4326  min=-6.8762  max=+8.2521
    qed    mean=+0.7323  std=0.1382  min=+0.1166  max=+0.9484
    SAS    mean=+3.0495  std=0.8353  min=+1.1327  max=+7.2893


{'qm9': {'mu': (2.649686336517334, 1.4013046026229858),
  'alpha': (75.22818756103516, 8.176578521728516),
  'homo': (-0.2401711791753769, 0.02181021124124527),
  'lumo': (0.011447526514530182, 0.04694028198719025),
  'gap': (0.25161856412887573, 0.04720733314752579),
  'cv': (31.59463882446289, 4.06702184677124)},
 'zinc': {'logP': (2.45540714263916, 1.4326330423355103),
  'qed': (0.7323237061500549, 0.1382189840078354),
  'SAS': (3.049480676651001, 0.835345983505249)}}

## 5 · Dataset + padded batch
`MoleculeDataset` + `collate_dense` pad variable-N graphs and emit a node mask — training-ready.

In [7]:
from torch.utils.data import DataLoader

ds = MoleculeDataset.from_loader(zinc)
loader = DataLoader(ds, batch_size=4, shuffle=True, collate_fn=collate_dense)
batch = next(iter(loader))
print({k: tuple(v.shape) for k, v in batch.items()})

{'X': (4, 27, 11), 'E': (4, 27, 27, 4), 'y': (4, 3), 'mask': (4, 27)}
